# Survival algorithm comparison


In [ ]:
# Shared configuration and output paths.
from pathlib import Path
import hashlib
import json
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SHARED_NOTEBOOK = os.environ.get(
    "CHD_SHARED_NOTEBOOK",
    str(Path.cwd() / "00_shared_config.ipynb"),
)
if not Path(SHARED_NOTEBOOK).exists():
    raise FileNotFoundError(
        f"Shared notebook not found: {SHARED_NOTEBOOK}. Place this notebook beside "
        "00_shared_config.ipynb, or set CHD_SHARED_NOTEBOOK."
    )
get_ipython().run_line_magic("run", f'"{SHARED_NOTEBOOK}"')

SERVER_SENSITIVITY_OUTPUT_DIR = Path(
    os.environ.get("CHD_SENSITIVITY_OUTPUT_DIR", SENSITIVITY_OUTPUT_DIR)
)
RESULTS_OUTPUT_DIR = SERVER_SENSITIVITY_OUTPUT_DIR / "04_results"
FIGURE_DATA_DIR = RESULTS_OUTPUT_DIR / "figure_source_data"
FIGURE_DIR = RESULTS_OUTPUT_DIR / "figures"
RESULT3_ROOT = RESULTS_OUTPUT_DIR / "result3_cox_vs_elasticnet_ipcw"
RESULT3_CACHE_DIR = RESULT3_ROOT / "elasticnet_cache"
RESULT4_UNO_DIR = RESULTS_OUTPUT_DIR / "result4_all_models_uno_ipcw"
FIGURE_DATA_DIR = RESULT4_UNO_DIR / "source_data"
FIGURE_DIR = RESULT4_UNO_DIR / "figures"

for directory in [RESULTS_OUTPUT_DIR, FIGURE_DATA_DIR, FIGURE_DIR, RESULT4_UNO_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Shared config:", SHARED_NOTEBOOK)
print("Unified Uno output:", RESULT4_UNO_DIR)


In [ ]:
# 1. Analysis specification, labels, and CNS palette
from joblib import Parallel, delayed
from sksurv.metrics import concordance_index_ipcw
from sksurv.nonparametric import CensoringDistributionEstimator
from sksurv.util import Surv

try:
    from sksurv.metrics import _estimate_concordance_index
except ImportError:
    _estimate_concordance_index = None

CNS = {
    "blue": "#3C5488",
    "red": "#E64B35",
    "green": "#00A087",
    "teal": "#4DBBD5",
    "orange": "#F39B7F",
    "purple": "#8491B4",
    "gray": "#6E6E6E",
    "light_gray": "#D9D9D9",
    "black": "#222222",
}

REPRESENTATIONS = [
    ("NMR_score", "Clinical_plus_PRS_plus_NMR_score"),
    ("Spearman-pruned NMR", "Clinical_plus_PRS_plus_SpearmanNMR"),
    ("Raw106 NMR", "Clinical_plus_PRS_plus_Raw106NMR"),
]
REP_ORDER = [label for label, _ in REPRESENTATIONS]
FEATURE_TO_REP = {feature: label for label, feature in REPRESENTATIONS}

SAVED_MODELS = ["Cox PH", "Elastic-net Cox", "XGBoost Cox", "DeepSurv"]
MODEL_ORDER = ["Cox PH", "Elastic-net Cox", "XGBoost survival", "DeepSurv"]
MODEL_COLORS = {
    "Cox PH": CNS["black"],
    "Elastic-net Cox": CNS["green"],
    "XGBoost survival": CNS["blue"],
    "DeepSurv": CNS["red"],
}
MODEL_MARKERS = {
    "Cox PH": "o",
    "Elastic-net Cox": "D",
    "XGBoost survival": "s",
    "DeepSurv": "^",
}

UNO_TAU = float(os.environ.get("CHD_RESULT4_UNO_TAU", "10"))
BOOTSTRAP_N = int(os.environ.get("CHD_RESULT4_UNO_BOOTSTRAP_N", "1000"))
BOOTSTRAP_SEED = int(os.environ.get("CHD_RESULT4_UNO_BOOTSTRAP_SEED", "2026"))
N_JOBS = int(os.environ.get("CHD_RESULT4_UNO_N_JOBS", str(CPU_WORKERS)))
if N_JOBS < 1:
    raise ValueError("CHD_RESULT4_UNO_N_JOBS must be at least 1")

if not np.isclose(UNO_TAU, 10.0):
    warnings.warn(f"The manuscript specification is 10 years, but UNO_TAU={UNO_TAU:g} was requested.")
if BOOTSTRAP_N < 1000:
    warnings.warn("Fewer than 1,000 bootstrap replicates were requested; use 1,000 for final manuscript inference.")

plt.rcParams.update({
    "figure.dpi": 170,
    "savefig.dpi": 600,
    "font.family": "DejaVu Sans",
    "font.size": 13.0,
    "axes.titlesize": 15.0,
    "axes.labelsize": 13.5,
    "xtick.labelsize": 12.0,
    "ytick.labelsize": 12.0,
    "legend.fontsize": 11.0,
    "axes.linewidth": 0.9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

def clean_axes(ax, grid_axis="x"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, axis=grid_axis, color=CNS["light_gray"], linewidth=0.7, alpha=0.85)
    ax.tick_params(length=3.5, width=0.8, color=CNS["black"])

def save_figure(fig, stem):
    outputs = []
    for extension in ["png", "pdf", "svg"]:
        path = FIGURE_DIR / f"{stem}.{extension}"
        fig.savefig(path, dpi=600, bbox_inches="tight", facecolor="white")
        outputs.append(path)
    print("Saved:", stem)
    return outputs

def offsets_for(labels, span=0.54):
    labels = list(labels)
    if len(labels) <= 1:
        return {labels[0]: 0.0} if labels else {}
    return dict(zip(labels, np.linspace(-span / 2, span / 2, len(labels))))

print(f"Uno IPCW tau={UNO_TAU:g} years | bootstrap={BOOTSTRAP_N} | n_jobs={N_JOBS}")


In [ ]:
# Train Result 4 algorithms. Hyperparameters use validation only; the test split is untouched until final evaluation.
import itertools, random, xgboost as xgb
import torch
from torch import nn
from sksurv.metrics import concordance_index_ipcw
from sklearn.exceptions import ConvergenceWarning as SklearnConvergenceWarning
from sksurv.util import Surv

RESULT4_FEATURES=["Clinical_plus_PRS_plus_NMR_score","Clinical_plus_PRS_plus_SpearmanNMR","Clinical_plus_PRS_plus_Raw106NMR"]
df_train,df_val,df_test,final_feature_sets,pipeline_manifest=validate_pipeline_artifacts(load_frames=True)
_result4_core_sets={name:final_feature_sets[name] for name in RESULT4_FEATURES}
load_all_risk_records(_result4_core_sets,["Cox PH","Elastic-net Cox"])
XGB_TRIALS=int(os.environ.get("CHD_XGB_TRIALS","24")); DEEPSURV_TRIALS=int(os.environ.get("CHD_DEEPSURV_TRIALS","24"))
GPU_REQUIRED=os.environ.get("CHD_GPU_REQUIRED","1")=="1"
if GPU_REQUIRED and not torch.cuda.is_available(): raise RuntimeError("Result 4 requires a visible CUDA GPU; GPU preflight failed")
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Result 4 GPU preflight:",DEVICE,torch.cuda.get_device_name(0) if DEVICE.type=="cuda" else "CPU override",flush=True)
if DEVICE.type=="cuda":
    _preflight=xgb.DMatrix(np.asarray([[0.0],[1.0],[2.0],[3.0]],dtype=np.float32),label=np.asarray([1.0,-2.0,3.0,-4.0]))
    xgb.train({"objective":"survival:cox","tree_method":"hist","device":"cuda","verbosity":0},_preflight,num_boost_round=1)
    print("XGBoost CUDA preflight passed",flush=True)

def _surv(frame): return Surv.from_arrays(event=frame.Event.astype(bool).to_numpy(),time=frame.Time.astype(float).to_numpy())
def _uno_val(risk): return float(concordance_index_ipcw(_surv(df_train),_surv(df_val),np.asarray(risk,float),tau=10.0)[0])
def _xgb_label(frame): return np.where(frame.Event.astype(bool),frame.Time.astype(float),-frame.Time.astype(float))

def _resume_result4_model(model_name,feature_set,code_tag,expected_trials):
    metadata_path=Path(risk_metadata_path(model_name,feature_set))
    if not metadata_path.exists(): return None
    metadata=json.loads(metadata_path.read_text(encoding="utf-8")); payload=metadata.get("signature_payload",{}); hyper=payload.get("hyperparameters",{})
    if metadata.get("signature")!=canonical_json_digest(payload): return None
    if payload.get("pipeline_manifest_sha256")!=current_pipeline_sha256() or payload.get("code_tag")!=code_tag: return None
    if int(hyper.get("trials",-1))!=int(expected_trials) or list(payload.get("ordered_feature_columns",[]))!=list(final_feature_sets[feature_set]): return None
    load_all_risk_records({feature_set:final_feature_sets[feature_set]},[model_name])
    log_progress(f"RESUME Result4 {model_name} | {feature_set} | signed risk cache valid")
    return hyper

def train_xgboost_feature_set(feature_set):
    resumed=_resume_result4_model("XGBoost Cox",feature_set,"xgboost_cox_gpu_validation_v2",XGB_TRIALS)
    if resumed is not None: return {"Feature_Set":feature_set,"Model":"XGBoost survival","Validation_Uno_C_10y":resumed.get("validation_uno_c_10y"),"Best_Iteration":resumed.get("num_boost_round"),"Trial_N":resumed.get("trials"),"Cache_Status":"resumed"}
    X_train,X_val,X_test,columns=prepare_X_train_val_test(df_train,df_val,df_test,final_feature_sets[feature_set],scale=False)
    base={"objective":"survival:cox","eval_metric":"cox-nloglik","tree_method":"hist","device":"cuda" if DEVICE.type=="cuda" else "cpu","seed":RANDOM_STATE,"nthread":INNER_THREADS}
    dtrain=xgb.DMatrix(X_train,label=_xgb_label(df_train),feature_names=columns); dval=xgb.DMatrix(X_val,label=_xgb_label(df_val),feature_names=columns); dtest=xgb.DMatrix(X_test,feature_names=columns)
    grid=list(itertools.product([0.01,0.03,0.06],[2,3,4],[0.7,0.9],[0.7,0.9],[0.0,0.5],[1.0,5.0]))
    rng=random.Random(RANDOM_STATE+sum(map(ord,feature_set))); rng.shuffle(grid); trials=grid[:min(XGB_TRIALS,len(grid))]
    rows=[]; best=None
    for trial,(eta,depth,subsample,colsample,min_child,reg_lambda) in enumerate(trials,1):
        params={**base,"eta":eta,"max_depth":depth,"subsample":subsample,"colsample_bytree":colsample,"min_child_weight":min_child,"lambda":reg_lambda}
        log_progress(f"Result4 XGBoost {feature_set} trial={trial}/{len(trials)}")
        booster=xgb.train(params,dtrain,num_boost_round=2000,evals=[(dval,"validation")],early_stopping_rounds=75,verbose_eval=False)
        risk=booster.predict(dval,iteration_range=(0,booster.best_iteration+1)); score=_uno_val(risk)
        row={"Feature_Set":feature_set,"Trial":trial,"Validation_Uno_C_10y":score,"Best_Iteration":int(booster.best_iteration),**{k:params[k] for k in ["eta","max_depth","subsample","colsample_bytree","min_child_weight","lambda"]}}; rows.append(row)
        if best is None or score>best[0]: best=(score,params,int(booster.best_iteration)+1)
    if best is None: raise RuntimeError(f"No valid XGBoost trial for {feature_set}")
    final=xgb.train(best[1],dtrain,num_boost_round=best[2],verbose_eval=False)
    risks={"train_risk":final.predict(dtrain),"val_risk":final.predict(dval),"test_risk":final.predict(dtest)}
    hyper={**{k:v for k,v in best[1].items() if k not in {"nthread"}},"num_boost_round":best[2],"validation_uno_c_10y":best[0],"trials":len(trials),"test_used_for_tuning":False}
    save_risk_record("XGBoost Cox",feature_set,risks,columns,hyper,"xgboost_cox_gpu_validation_v2")
    atomic_write_csv(pd.DataFrame(rows),os.path.join(AUDIT_DIR,f"xgboost_{safe_filename(feature_set)}_validation_trials.csv"),index=False)
    final.save_model(os.path.join(CHECKPOINT_DIR,f"xgboost_{safe_filename(feature_set)}.json"))
    return {"Feature_Set":feature_set,"Model":"XGBoost survival","Validation_Uno_C_10y":best[0],"Best_Iteration":best[2],"Trial_N":len(trials)}

class StandardDeepSurv(nn.Module):
    def __init__(self,input_dim,hidden,dropout):
        super().__init__(); layers=[]; previous=input_dim
        for width in hidden:
            layers.extend([nn.Linear(previous,width),nn.BatchNorm1d(width),nn.ReLU(),nn.Dropout(dropout)]); previous=width
        layers.append(nn.Linear(previous,1)); self.network=nn.Sequential(*layers)
    def forward(self,x): return self.network(x).reshape(-1)

def breslow_negative_partial_loglik(risk,time,event):
    order=torch.argsort(time,descending=True); risk=risk[order]; time=time[order]; event=event[order]
    _,counts=torch.unique_consecutive(time,return_counts=True); ends=torch.cumsum(counts,dim=0)-1
    log_denominator=torch.repeat_interleave(torch.logcumsumexp(risk,dim=0)[ends],counts)
    return -((risk-log_denominator)*event).sum()/torch.clamp(event.sum(),min=1.0)

def train_deepsurv_feature_set(feature_set):
    resumed=_resume_result4_model("DeepSurv",feature_set,"standard_deepsurv_breslow_gpu_v2",DEEPSURV_TRIALS)
    if resumed is not None: return {"Feature_Set":feature_set,"Model":"DeepSurv","Validation_Uno_C_10y":resumed.get("validation_uno_c_10y"),"Best_Epoch":resumed.get("best_epoch"),"Trial_N":resumed.get("trials"),"Cache_Status":"resumed"}
    X_train,X_val,X_test,columns=prepare_X_train_val_test(df_train,df_val,df_test,final_feature_sets[feature_set],scale=True)
    tensors=[torch.as_tensor(frame.to_numpy(np.float32),device=DEVICE) for frame in [X_train,X_val,X_test]]
    train_time=torch.as_tensor(df_train.Time.to_numpy(np.float32),device=DEVICE); train_event=torch.as_tensor(df_train.Event.to_numpy(np.float32),device=DEVICE)
    configs=list(itertools.product([(64,32),(128,64),(256,128),(128,64,32)],[0.0,0.15,0.30],[1e-4,3e-4,1e-3],[0.0,1e-5,1e-4]))
    rng=random.Random(RANDOM_STATE+17+sum(map(ord,feature_set))); rng.shuffle(configs); configs=configs[:min(DEEPSURV_TRIALS,len(configs))]
    rows=[]; best=None
    for trial,(hidden,dropout,lr,weight_decay) in enumerate(configs,1):
        torch.manual_seed(RANDOM_STATE+trial); torch.cuda.manual_seed_all(RANDOM_STATE+trial)
        model=StandardDeepSurv(len(columns),hidden,dropout).to(DEVICE); optimizer=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=weight_decay)
        best_state=None; best_score=-np.inf; best_epoch=0; stale=0
        for epoch in range(1,401):
            model.train(); optimizer.zero_grad(set_to_none=True); loss=breslow_negative_partial_loglik(model(tensors[0]),train_time,train_event)
            if not torch.isfinite(loss): raise RuntimeError(f"Non-finite DeepSurv loss: {feature_set} trial {trial}")
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); optimizer.step()
            if epoch%5==0:
                model.eval()
                with torch.no_grad(): val_risk=model(tensors[1]).detach().cpu().numpy()
                score=_uno_val(val_risk)
                if score>best_score+1e-5: best_score=score; best_epoch=epoch; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; stale=0
                else: stale+=1
                if stale>=12: break
            if epoch%25==0: log_progress(f"Result4 DeepSurv {feature_set} trial={trial}/{len(configs)} epoch={epoch} loss={float(loss):.6f}")
        if best_state is None: raise RuntimeError("DeepSurv validation never completed")
        rows.append({"Feature_Set":feature_set,"Trial":trial,"Hidden":"-".join(map(str,hidden)),"Dropout":dropout,"Learning_Rate":lr,"Weight_Decay":weight_decay,"Best_Epoch":best_epoch,"Validation_Uno_C_10y":best_score})
        if best is None or best_score>best[0]: best=(best_score,hidden,dropout,lr,weight_decay,best_epoch,best_state)
        del model; torch.cuda.empty_cache() if DEVICE.type=="cuda" else None
    score,hidden,dropout,lr,weight_decay,best_epoch,state=best
    final=StandardDeepSurv(len(columns),hidden,dropout).to(DEVICE); final.load_state_dict(state); final.eval()
    with torch.no_grad(): values=[final(tensor).detach().cpu().numpy() for tensor in tensors]
    risks=dict(zip(["train_risk","val_risk","test_risk"],values))
    hyper={"architecture":"standard single-branch multilayer perceptron","hidden":list(hidden),"dropout":dropout,"learning_rate":lr,"weight_decay":weight_decay,"best_epoch":best_epoch,"validation_uno_c_10y":score,"trials":len(configs),"test_used_for_tuning":False}
    save_risk_record("DeepSurv",feature_set,risks,columns,hyper,"standard_deepsurv_breslow_gpu_v2")
    torch.save({"state_dict":state,"columns":columns,"hyperparameters":hyper},os.path.join(CHECKPOINT_DIR,f"deepsurv_{safe_filename(feature_set)}.pt"))
    atomic_write_csv(pd.DataFrame(rows),os.path.join(AUDIT_DIR,f"deepsurv_{safe_filename(feature_set)}_validation_trials.csv"),index=False)
    return {"Feature_Set":feature_set,"Model":"DeepSurv","Validation_Uno_C_10y":score,"Best_Epoch":best_epoch,"Trial_N":len(configs)}

# XGBoost and DeepSurv share one GPU and therefore run sequentially by default; each reports live progress.
xgb_selection=[]; deepsurv_selection=[]
for _feature in RESULT4_FEATURES: xgb_selection.append(train_xgboost_feature_set(_feature))
for _feature in RESULT4_FEATURES: deepsurv_selection.append(train_deepsurv_feature_set(_feature))
atomic_write_csv(pd.DataFrame(xgb_selection+deepsurv_selection),os.path.join(TABLE_DIR,"result4_validation_model_selection.csv"),index=False)


In [ ]:
# Load only EID-signed predictions from the locked pipeline population.
df_train,df_val,df_test,final_feature_sets,pipeline_manifest=validate_pipeline_artifacts(load_frames=True)
REPRESENTATIONS=[("NMR_score","Clinical_plus_PRS_plus_NMR_score"),("Spearman-pruned NMR","Clinical_plus_PRS_plus_SpearmanNMR"),("Raw106 NMR","Clinical_plus_PRS_plus_Raw106NMR")]
REP_ORDER=[label for label,_ in REPRESENTATIONS]; FEATURE_TO_REP={feature:label for label,feature in REPRESENTATIONS}
FORBIDDEN_RESULT4_FEATURE_SET="Clinical_plus_PRS_plus_Full249NMR"
if FORBIDDEN_RESULT4_FEATURE_SET in [feature for _,feature in REPRESENTATIONS]:
    raise RuntimeError("Expanded Full249 panel is Result 3 sensitivity only and must not enter Result 4")
_model_names=["Cox PH","Elastic-net Cox","XGBoost Cox","DeepSurv"]
MODEL_ORDER=["Cox PH","Elastic-net Cox","XGBoost survival","DeepSurv"]
selected_feature_sets={name:final_feature_sets[name] for _,name in REPRESENTATIONS}
_signed=load_all_risk_records(selected_feature_sets,_model_names)
risk_map={}; train_risk_map={}; val_risk_map={}; audit=[]
for representation,feature_set in REPRESENTATIONS:
    for saved,display_name in zip(_model_names,MODEL_ORDER):
        record=_signed[(MAIN_SCENARIO_NAME,feature_set,saved)]
        train_risk_map[(representation,display_name)]=np.asarray(record["train_risk"],dtype=float)
        val_risk_map[(representation,display_name)]=np.asarray(record["val_risk"],dtype=float)
        risk_map[(representation,display_name)]=np.asarray(record["test_risk"],dtype=float)
        if len(train_risk_map[(representation,display_name)])!=len(df_train) or len(val_risk_map[(representation,display_name)])!=len(df_val) or len(risk_map[(representation,display_name)])!=len(df_test):
            raise RuntimeError(f"Risk/data length mismatch: {representation} | {display_name}")
        audit.append({"Representation":representation,"Feature_Set":feature_set,"Model":display_name,"Saved_Model_Label":saved,"Evaluation_Population":"locked test","N":len(df_test),"Test_Eid_SHA256":eid_digest(df_test),"Source":"EID-signed risk record","Pipeline_Manifest_SHA256":current_pipeline_sha256()})
risk_audit_df=pd.DataFrame(audit); risk_audit_df.to_csv(RESULT4_UNO_DIR/"risk_alignment_audit.csv",index=False)

train_digest=dataframe_digest(df_train,["Eid","Time","Event"])
test_digest=dataframe_digest(df_test,["Eid","Time","Event"])


In [ ]:
# 3. Compute or load unified Uno IPCW C with one paired bootstrap design
model_keys = [(representation, model) for representation in REP_ORDER for model in MODEL_ORDER]
risk_matrix = np.column_stack([risk_map[key] for key in model_keys])

test_event = df_test["Event"].astype(bool).to_numpy()
test_time = df_test["Time"].astype(float).to_numpy()
y_train = Surv.from_arrays(
    event=df_train["Event"].astype(bool).to_numpy(),
    time=df_train["Time"].astype(float).to_numpy(),
)
y_test = Surv.from_arrays(event=test_event, time=test_time)

# Estimate the censoring distribution once from training. Resampling these
# subject-specific weights is equivalent to refitting only the test empirical
# distribution while keeping the prespecified training censoring estimator.
censoring = CensoringDistributionEstimator().fit(y_train)
tau_mask = test_time < UNO_TAU
ipcw = np.zeros(len(df_test), dtype=float)
ipcw[tau_mask] = censoring.predict_ipcw(y_test[tau_mask])
uno_weights = np.square(ipcw)

def public_uno(risk):
    return float(concordance_index_ipcw(y_train, y_test, risk, tau=UNO_TAU)[0])

def weighted_uno(event, time, risk, weights):
    if _estimate_concordance_index is None:
        frame = Surv.from_arrays(event=event.astype(bool), time=time.astype(float))
        return float(concordance_index_ipcw(y_train, frame, risk, tau=UNO_TAU)[0])
    return float(_estimate_concordance_index(event, time, risk, weights)[0])

# Validate the optimized common-weight route against the public scikit-survival API.
for column_index in [0, risk_matrix.shape[1] - 1]:
    public_value = public_uno(risk_matrix[:, column_index])
    weighted_value = weighted_uno(test_event, test_time, risk_matrix[:, column_index], uno_weights)
    if not np.isclose(public_value, weighted_value, atol=1e-12, rtol=1e-10):
        raise RuntimeError(
            f"Uno implementation validation failed: public={public_value}, optimized={weighted_value}"
        )

signature_payload = {
    "test_sha256": test_digest,
    "train_sha256": train_digest,
    "tau": UNO_TAU,
    "bootstrap_n": BOOTSTRAP_N,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "model_keys": [list(key) for key in model_keys],
    "risk_sha256": hashlib.sha256(np.asarray(risk_matrix, dtype=np.float64).tobytes()).hexdigest(),
}
signature = hashlib.sha256(json.dumps(signature_payload, sort_keys=True).encode("utf-8")).hexdigest()
summary_path = RESULT4_UNO_DIR / "result4_all_models_uno_ipcw_summary.csv"
bootstrap_path = RESULT4_UNO_DIR / "result4_all_models_uno_ipcw_bootstrap.csv.gz"
metadata_path = RESULT4_UNO_DIR / "result4_all_models_uno_ipcw_metadata.json"

cache_valid = False
if summary_path.exists() and bootstrap_path.exists() and metadata_path.exists():
    with metadata_path.open("r", encoding="utf-8") as handle:
        cached_metadata = json.load(handle)
    cache_valid = cached_metadata.get("signature") == signature

if cache_valid:
    print("Loading matching unified Uno bootstrap cache.")
    uno_summary = pd.read_csv(summary_path)
    uno_bootstrap = pd.read_csv(bootstrap_path)
else:
    print("Computing unified paired Uno bootstrap. No prediction model is being fitted.", flush=True)
    point_values = np.asarray([
        weighted_uno(test_event, test_time, risk_matrix[:, column], uno_weights)
        for column in range(risk_matrix.shape[1])
    ])

    def one_bootstrap(seed):
        rng = np.random.default_rng(int(seed))
        index = rng.integers(0, len(df_test), size=len(df_test))
        event = test_event[index]
        if np.unique(event).size < 2:
            return np.full(risk_matrix.shape[1], np.nan)
        time = test_time[index]
        weights = uno_weights[index]
        risks = risk_matrix[index, :]
        try:
            return np.asarray([
                weighted_uno(event, time, risks[:, column], weights)
                for column in range(risks.shape[1])
            ], dtype=float)
        except Exception:
            return np.full(risk_matrix.shape[1], np.nan)

    seed_sequence = np.random.SeedSequence(BOOTSTRAP_SEED)
    bootstrap_seeds = [int(child.generate_state(1)[0]) for child in seed_sequence.spawn(BOOTSTRAP_N)]
    bootstrap_values = Parallel(
        n_jobs=N_JOBS,
        backend="loky",
        verbose=10,
        max_nbytes="32M",
    )(delayed(one_bootstrap)(seed) for seed in bootstrap_seeds)
    bootstrap_values = np.vstack(bootstrap_values)

    column_names = [f"{representation}__{model}" for representation, model in model_keys]
    uno_bootstrap = pd.DataFrame(bootstrap_values, columns=column_names)
    uno_bootstrap.insert(0, "Bootstrap_Replicate", np.arange(1, len(uno_bootstrap) + 1))
    uno_bootstrap.to_csv(bootstrap_path, index=False, compression="gzip")

    summary_rows = []
    for column, (representation, model) in enumerate(model_keys):
        values = bootstrap_values[:, column]
        valid = values[np.isfinite(values)]
        summary_rows.append({
            "Representation": representation,
            "Feature_Set": dict(REPRESENTATIONS)[representation],
            "Model": model,
            "Split": "test",
            "N_Test": len(df_test),
            "Tau_Years": UNO_TAU,
            "Uno_C": point_values[column],
            "Uno_C_Lower95": np.quantile(valid, 0.025),
            "Uno_C_Upper95": np.quantile(valid, 0.975),
            "Bootstrap_N_Requested": BOOTSTRAP_N,
            "Bootstrap_N_Used": len(valid),
        })
    uno_summary = pd.DataFrame(summary_rows)
    uno_summary.to_csv(summary_path, index=False)
    with metadata_path.open("w", encoding="utf-8") as handle:
        json.dump({"signature": signature, "signature_payload": signature_payload}, handle, indent=2)

print("Uno summary:", summary_path)
print("Uno bootstrap:", bootstrap_path)
display(uno_summary)


In [ ]:
# 4. Derive paired differences relative to Cox PH from the same bootstrap rows
bootstrap_values = uno_bootstrap.drop(columns=["Bootstrap_Replicate"], errors="ignore")
delta_rows = []
for representation in REP_ORDER:
    cox_column = f"{representation}__Cox PH"
    cox_point = float(uno_summary.loc[
        (uno_summary["Representation"] == representation) & (uno_summary["Model"] == "Cox PH"),
        "Uno_C",
    ].iloc[0])
    for model in MODEL_ORDER[1:]:
        model_column = f"{representation}__{model}"
        model_point = float(uno_summary.loc[
            (uno_summary["Representation"] == representation) & (uno_summary["Model"] == model),
            "Uno_C",
        ].iloc[0])
        delta = pd.to_numeric(bootstrap_values[model_column], errors="coerce") - pd.to_numeric(
            bootstrap_values[cox_column], errors="coerce"
        )
        delta = delta[np.isfinite(delta)]
        delta_rows.append({
            "Representation": representation,
            "Model": model,
            "Reference_Model": "Cox PH",
            "Split": "test",
            "Tau_Years": UNO_TAU,
            "Uno_C_Model": model_point,
            "Uno_C_Cox": cox_point,
            "Delta_Uno_C_vs_Cox": model_point - cox_point,
            "Delta_Uno_C_Lower95": np.quantile(delta, 0.025),
            "Delta_Uno_C_Upper95": np.quantile(delta, 0.975),
            "Bootstrap_N_Used": len(delta),
        })

uno_delta = pd.DataFrame(delta_rows)
delta_path = RESULT4_UNO_DIR / "result4_all_models_delta_uno_ipcw_vs_cox.csv"
uno_delta.to_csv(delta_path, index=False)
print("Paired Uno differences:", delta_path)
display(uno_delta)


In [ ]:
# Figure 4A: absolute test-set Uno IPCW C for all four algorithms
offsets = offsets_for(MODEL_ORDER)
y = np.arange(len(REP_ORDER))
fig, ax = plt.subplots(figsize=(9.0, 5.2))
for model in MODEL_ORDER:
    sub = uno_summary.loc[uno_summary["Model"] == model].copy()
    sub["Order"] = sub["Representation"].map({rep: index for index, rep in enumerate(REP_ORDER)})
    sub = sub.sort_values("Order")
    x = pd.to_numeric(sub["Uno_C"], errors="coerce").to_numpy()
    lo = pd.to_numeric(sub["Uno_C_Lower95"], errors="coerce").to_numpy()
    hi = pd.to_numeric(sub["Uno_C_Upper95"], errors="coerce").to_numpy()
    ax.errorbar(
        x,
        y + offsets[model],
        xerr=np.vstack([x - lo, hi - x]),
        fmt=MODEL_MARKERS[model],
        color=MODEL_COLORS[model],
        ecolor=MODEL_COLORS[model],
        markersize=6,
        capsize=3,
        elinewidth=1.2,
        label=model,
    )
ax.set_yticks(y)
ax.set_yticklabels(REP_ORDER)
ax.invert_yaxis()
ax.set_xlabel(f"Uno IPCW C-index at {UNO_TAU:g} years (95% CI)")
ax.set_title("Figure 4A. Discrimination across survival algorithms", loc="left", fontweight="bold")
clean_axes(ax, "x")
ax.legend(frameon=False, ncol=2, loc="best")
fig.tight_layout()
save_figure(fig, "figure4A_all_models_uno_ipcw_cns_final")
plt.show()


In [ ]:
# Figure 4B: paired difference in Uno IPCW discrimination relative to Cox PH
comparison_models = MODEL_ORDER[1:]
offsets = offsets_for(comparison_models, span=0.44)
y = np.arange(len(REP_ORDER))
fig, ax = plt.subplots(figsize=(9.2, 5.2))
for model in comparison_models:
    sub = uno_delta.loc[uno_delta["Model"] == model].copy()
    sub["Order"] = sub["Representation"].map({rep: index for index, rep in enumerate(REP_ORDER)})
    sub = sub.sort_values("Order")
    x = pd.to_numeric(sub["Delta_Uno_C_vs_Cox"], errors="coerce").to_numpy()
    lo = pd.to_numeric(sub["Delta_Uno_C_Lower95"], errors="coerce").to_numpy()
    hi = pd.to_numeric(sub["Delta_Uno_C_Upper95"], errors="coerce").to_numpy()
    ax.errorbar(
        x,
        y + offsets[model],
        xerr=np.vstack([x - lo, hi - x]),
        fmt=MODEL_MARKERS[model],
        color=MODEL_COLORS[model],
        ecolor=MODEL_COLORS[model],
        markersize=6,
        capsize=3,
        elinewidth=1.3,
        label=model,
    )
ax.axvline(0, color=CNS["gray"], linestyle="--", linewidth=1.0)
ax.set_yticks(y)
ax.set_yticklabels(REP_ORDER)
ax.invert_yaxis()
ax.set_xlabel(f"Difference in Uno IPCW C-index relative to Cox PH at {UNO_TAU:g} years (95% CI)")
ax.set_title("Figure 4B. Difference in discrimination relative to Cox", loc="left", fontweight="bold")
clean_axes(ax, "x")
ax.legend(frameon=False, ncol=2, loc="best")
fig.tight_layout()
save_figure(fig, "figure4B_difference_in_discrimination_relative_to_cox_uno_cns_final")
plt.show()


In [ ]:
def _pipeline_context(manifest):
    nested = manifest.get("analysis_context", {}) if isinstance(manifest, dict) else {}
    cohort_id = str(manifest.get("cohort_id", nested.get("cohort_id", "unspecified")))
    analysis_role = str(manifest.get("analysis_role", nested.get("analysis_role", "unspecified")))
    if cohort_id.lower() == "coverage90" and analysis_role.lower() == "primary":
        raise RuntimeError("coverage90 must remain a sensitivity cohort; pipeline manifest labels it primary")
    return cohort_id, analysis_role

# Unified secondary 10-year metrics from the same 12 locked risk records.
import re
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.exceptions import ConvergenceWarning
from dcurves import dca as dcurves_dca

RESULT4_HORIZON = 10.0
RESULT4_IBS_GRID_N = int(os.environ.get("CHD_RESULT4_IBS_GRID_N", "51"))
RESULT4_DCA_THRESHOLDS = np.arange(0.01, 0.201, 0.005)

def _r4_fit_calibrator(raw_risk, label):
    raw_risk = np.asarray(raw_risk, dtype=float).reshape(-1)

    if not np.isfinite(raw_risk).all():
        raise RuntimeError(f"Non-finite training risk values: {label}")

    center = float(raw_risk.mean())
    scale = float(raw_risk.std(ddof=0))

    if not np.isfinite(scale) or scale <= 0:
        raise RuntimeError(f"Degenerate training risk: {label}")

    frame = pd.DataFrame({
        "Time": df_train["Time"].to_numpy(float),
        "Event": df_train["Event"].to_numpy(int),
        "Risk_Score": (raw_risk - center) / scale,
    })

    attempts = [
        (0.0, 0.50),
        (0.0, 0.25),
        (1e-8, 0.50),
        (1e-6, 0.50),
        (1e-4, 0.50),
    ]

    errors = []

    for penalizer, step_size in attempts:
        model = CoxPHFitter(
            penalizer=penalizer,
            l1_ratio=0.0,
        )

        try:
            with warnings.catch_warnings():
                warnings.simplefilter("error", ConvergenceWarning)

                model.fit(
                    frame,
                    duration_col="Time",
                    event_col="Event",
                    show_progress=False,
                    batch_mode=True,
                    fit_options={
                        "step_size": step_size,
                        "max_steps": 1000,
                        "precision": 1e-7,
                    },
                )

            coefficient = float(model.params_.loc["Risk_Score"])
            if not np.isfinite(coefficient):
                raise RuntimeError("Non-finite calibration coefficient")

            log_progress(
                f"Calibration OK | {label} | "
                f"penalizer={penalizer:g} | "
                f"step_size={step_size:g} | "
                f"slope={coefficient:.6f}"
            )

            return {
                "model": model,
                "center": center,
                "scale": scale,
                "label": label,
                "penalizer": float(penalizer),
                "step_size": float(step_size),
                "calibration_slope": coefficient,
                "method": (
                    "training-only Cox recalibration of standardized model score; "
                    f"L2 penalizer={penalizer:g}"
                ),
            }

        except Exception as exc:
            errors.append(
                f"penalizer={penalizer:g}, "
                f"step_size={step_size:g}: "
                f"{type(exc).__name__}: {exc}"
            )

    raise RuntimeError(
        f"All calibration attempts failed: {label}\n"
        + "\n".join(errors)
    )

def _r4_predict_risk(calibrator, raw_risk, times):
    score = (np.asarray(raw_risk, dtype=float).reshape(-1) - calibrator["center"]) / calibrator["scale"]
    survival = calibrator["model"].predict_survival_function(
        pd.DataFrame({"Risk_Score": score}), times=np.asarray(times, dtype=float)
    )
    return np.clip(1.0 - survival.to_numpy(dtype=float).T, 0.0, 1.0)

def _r4_subject_ibs(calibrator, raw_risk):
    times = np.linspace(0.0, RESULT4_HORIZON, RESULT4_IBS_GRID_N)
    prediction = _r4_predict_risk(calibrator, raw_risk, times)
    followup = df_test["Time"].to_numpy(float); event = df_test["Event"].to_numpy(int)
    censoring_km = KaplanMeierFitter().fit(
        df_train["Time"].to_numpy(float), event_observed=1-df_train["Event"].to_numpy(int)
    )
    contributions = np.zeros_like(prediction)
    for column, point in enumerate(times):
        cases = (event == 1) & (followup <= point); controls = followup > point
        observed = cases.astype(float); weight = np.zeros(len(df_test), dtype=float)
        if cases.any():
            left_times = np.maximum(np.nextafter(followup[cases], -np.inf), 0.0)
            weight[cases] = 1.0 / np.clip(np.asarray(censoring_km.predict(left_times), float), 1e-8, 1.0)
        if controls.any():
            weight[controls] = 1.0 / float(np.clip(censoring_km.predict(point), 1e-8, 1.0))
        contributions[:, column] = weight * (observed - prediction[:, column]) ** 2
    integrate = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
    return integrate(contributions, times, axis=1) / RESULT4_HORIZON

def _r4_km_risk_ci(frame):
    km = KaplanMeierFitter(alpha=0.05).fit(frame["Time"], event_observed=frame["Event"])
    survival = float(km.predict(RESULT4_HORIZON)); ci = km.confidence_interval_survival_function_
    eligible = ci.loc[ci.index.astype(float) <= RESULT4_HORIZON]
    if eligible.empty:
        lower_survival = upper_survival = survival
    else:
        row = eligible.iloc[-1]
        lower_column = next((c for c in ci.columns if "lower" in str(c).lower()), ci.columns[0])
        upper_column = next((c for c in ci.columns if "upper" in str(c).lower()), ci.columns[-1])
        lower_survival = float(row[lower_column]); upper_survival = float(row[upper_column])
    return 1-survival, 1-upper_survival, 1-lower_survival

def _r4_calibration(prediction, representation, model):
    frame = df_test[["Time", "Event"]].reset_index(drop=True).copy()
    frame["Predicted_Risk"] = np.asarray(prediction, dtype=float)
    frame["Risk_Group"] = pd.qcut(frame["Predicted_Risk"].rank(method="first"), 10, labels=False) + 1
    rows = []
    for group, part in frame.groupby("Risk_Group", observed=True):
        observed, lower, upper = _r4_km_risk_ci(part)
        rows.append({
            "Representation": representation, "Model": model, "Split": "test",
            "Horizon_Years": RESULT4_HORIZON, "Risk_Group": int(group), "N": int(len(part)),
            "Event_N": int(part["Event"].sum()), "Predicted_Risk": float(part["Predicted_Risk"].mean()),
            "Observed_KM_Risk": observed, "Observed_KM_Risk_Lower95": lower,
            "Observed_KM_Risk_Upper95": upper,
        })
    return pd.DataFrame(rows)

def _r4_survival_dca(predictions, representation):
    data = df_test[["Time", "Event"]].reset_index(drop=True).copy(); model_columns=[]; labels={}
    for model, prediction in predictions.items():
        column = "model_" + re.sub(r"[^a-z0-9]+", "_", model.lower()).strip("_")
        data[column] = np.asarray(prediction, dtype=float); model_columns.append(column); labels[column] = model
    result = dcurves_dca(
        data=data, outcome="Event", modelnames=model_columns,
        thresholds=RESULT4_DCA_THRESHOLDS, time_to_outcome_col="Time", time=RESULT4_HORIZON,
    ).copy()
    result.columns = [str(column).lower() for column in result.columns]
    for source, target in [("model", "Strategy"), ("models", "Strategy"), ("label", "Strategy"),
                           ("threshold", "Threshold"), ("thresholds", "Threshold"),
                           ("net_benefit", "Net_Benefit"), ("net benefit", "Net_Benefit"), ("netbenefit", "Net_Benefit")]:
        if source in result.columns and target not in result.columns:
            result = result.rename(columns={source: target})
    if not {"Strategy", "Threshold", "Net_Benefit"}.issubset(result.columns):
        raise RuntimeError("Unexpected dcurves columns: " + ", ".join(result.columns))
    result["Strategy"] = result["Strategy"].replace(labels).replace({"all":"Treat all", "none":"Treat none", "All":"Treat all", "None":"Treat none"})
    result["Representation"] = representation; result["Horizon_Years"] = RESULT4_HORIZON
    result["Split"] = "test"; result["DCA_Method"] = "dcurves standard survival DCA; no external IPCW multiplication"
    return result

calibrators = {}; absolute_risk_10y = {}; subject_ibs = {}
for representation, model in model_keys:
    log_progress(f"Result4 secondary metrics | {representation} | {model}")
    calibrator = _r4_fit_calibrator(train_risk_map[(representation, model)], f"{representation} | {model}")
    calibrators[(representation, model)] = calibrator
    absolute_risk_10y[(representation, model)] = _r4_predict_risk(calibrator, risk_map[(representation, model)], [RESULT4_HORIZON])[:, 0]
    subject_ibs[(representation, model)] = _r4_subject_ibs(calibrator, risk_map[(representation, model)])

ibs_rows = []; delta_ibs_rows = []
for representation_index, representation in enumerate(REP_ORDER):
    for model_index, model in enumerate(MODEL_ORDER):
        values = subject_ibs[(representation, model)]
        rng = np.random.default_rng(BOOTSTRAP_SEED + 4000 + representation_index * 100 + model_index)
        bootstrap = np.asarray([values[rng.integers(0, len(values), len(values))].mean() for _ in range(BOOTSTRAP_N)])
        ibs_rows.append({
            "Representation": representation, "Feature_Set": dict(REPRESENTATIONS)[representation],
            "Model": model, "Split": "test", "Horizon_Years": RESULT4_HORIZON,
            "IPCW_IBS": float(values.mean()), "IPCW_IBS_Lower95": float(np.quantile(bootstrap, 0.025)),
            "IPCW_IBS_Upper95": float(np.quantile(bootstrap, 0.975)), "Bootstrap_N": BOOTSTRAP_N,
        })
        if model != "Cox PH":
            paired = values - subject_ibs[(representation, "Cox PH")]
            rng = np.random.default_rng(BOOTSTRAP_SEED + 4500 + representation_index * 100 + model_index)
            paired_bootstrap = np.asarray([paired[rng.integers(0, len(paired), len(paired))].mean() for _ in range(BOOTSTRAP_N)])
            delta_ibs_rows.append({
                "Representation": representation, "Model": model, "Reference_Model": "Cox PH",
                "Split": "test", "Horizon_Years": RESULT4_HORIZON,
                "Delta_IPCW_IBS_vs_Cox": float(paired.mean()),
                "Delta_IPCW_IBS_Lower95": float(np.quantile(paired_bootstrap, 0.025)),
                "Delta_IPCW_IBS_Upper95": float(np.quantile(paired_bootstrap, 0.975)),
                "Bootstrap_N": BOOTSTRAP_N,
            })

result4_ibs = pd.DataFrame(ibs_rows); result4_delta_ibs = pd.DataFrame(delta_ibs_rows)
result4_calibration = pd.concat([
    _r4_calibration(absolute_risk_10y[(representation, model)], representation, model)
    for representation, model in model_keys
], ignore_index=True)
calibration_summary_rows = []
for (representation, model, split, horizon), frame in result4_calibration.groupby(
    ["Representation", "Model", "Split", "Horizon_Years"], observed=True
):
    absolute_error = np.abs(frame["Observed_KM_Risk"] - frame["Predicted_Risk"])
    calibration_summary_rows.append({
        "Representation": representation, "Model": model, "Split": split, "Horizon_Years": horizon,
        "Weighted_Mean_Absolute_Calibration_Error": float(np.average(absolute_error, weights=frame["N"])),
        "Mean_Predicted_Risk": float(np.average(frame["Predicted_Risk"], weights=frame["N"])),
        "Mean_Observed_KM_Risk": float(np.average(frame["Observed_KM_Risk"], weights=frame["N"])),
    })
result4_calibration_summary = pd.DataFrame(calibration_summary_rows)
result4_dca = pd.concat([
    _r4_survival_dca({model:absolute_risk_10y[(representation,model)] for model in MODEL_ORDER}, representation)
    for representation in REP_ORDER
], ignore_index=True)

cohort_id, analysis_role = _pipeline_context(pipeline_manifest)
for frame in [result4_ibs, result4_delta_ibs, result4_calibration, result4_calibration_summary, result4_dca]:
    frame["Cohort_ID"] = cohort_id; frame["Analysis_Role"] = analysis_role
atomic_write_csv(result4_ibs, RESULT4_UNO_DIR/"result4_all_models_10y_ipcw_ibs.csv", index=False)
atomic_write_csv(result4_delta_ibs, RESULT4_UNO_DIR/"result4_all_models_delta_10y_ipcw_ibs_vs_cox.csv", index=False)
atomic_write_csv(result4_calibration, RESULT4_UNO_DIR/"result4_all_models_10y_calibration_by_decile.csv", index=False)
atomic_write_csv(result4_calibration_summary, RESULT4_UNO_DIR/"result4_all_models_10y_calibration_summary.csv", index=False)
atomic_write_csv(result4_dca, RESULT4_UNO_DIR/"result4_all_models_10y_survival_dca.csv", index=False)

# Separate CNS figures: no composite panels.
fig, ax = plt.subplots(figsize=(9.6, 6.0)); offsets = offsets_for(MODEL_ORDER)
for model in MODEL_ORDER:
    data = result4_ibs.loc[result4_ibs["Model"].eq(model)].set_index("Representation").loc[REP_ORDER]
    estimate = data["IPCW_IBS"].to_numpy(float); lower = data["IPCW_IBS_Lower95"].to_numpy(float); upper = data["IPCW_IBS_Upper95"].to_numpy(float)
    y = np.arange(len(REP_ORDER)) + offsets[model]
    ax.errorbar(estimate, y, xerr=[estimate-lower, upper-estimate], fmt=MODEL_MARKERS[model], color=MODEL_COLORS[model], capsize=3, label=model)
ax.set_yticks(np.arange(len(REP_ORDER))); ax.set_yticklabels(REP_ORDER); ax.invert_yaxis()
ax.set_xlabel("10-year IPCW integrated Brier score (95% CI; lower is better)")
ax.set_title("Supplementary Figure. All-model 10-year prediction error", loc="left", fontweight="bold")
ax.legend(frameon=False, ncol=2); clean_axes(ax); fig.tight_layout(); save_figure(fig, "supplement_result4_all_models_10y_ipcw_ibs"); plt.show()

for representation in REP_ORDER:
    fig, ax = plt.subplots(figsize=(8.6, 6.4))
    calibration = result4_calibration.loc[result4_calibration["Representation"].eq(representation)]
    for model in MODEL_ORDER:
        data = calibration.loc[calibration["Model"].eq(model)].sort_values("Predicted_Risk")
        ax.plot(data["Predicted_Risk"], data["Observed_KM_Risk"], marker=MODEL_MARKERS[model], linewidth=2.0, color=MODEL_COLORS[model], label=model)
    limit = float(max(0.01, calibration[["Predicted_Risk", "Observed_KM_Risk"]].to_numpy(float).max() * 1.05))
    ax.plot([0,limit],[0,limit], "--", color=CNS["gray"], linewidth=1.2, label="Ideal")
    ax.set_xlim(0,limit); ax.set_ylim(0,limit); ax.set_xlabel("Mean predicted CHD risk"); ax.set_ylabel("Kaplan-Meier observed CHD risk")
    ax.set_title(f"Supplementary calibration: {representation}, 10 years", loc="left", fontweight="bold")
    ax.grid(color=CNS["light_gray"], linewidth=0.8); ax.legend(frameon=False, fontsize=11.0); clean_axes(ax); fig.tight_layout()
    safe = re.sub(r"[^a-z0-9]+", "_", representation.lower()).strip("_")
    save_figure(fig, f"supplement_result4_{safe}_10y_calibration"); plt.show()

    fig, ax = plt.subplots(figsize=(8.8, 6.2)); dca_data = result4_dca.loc[result4_dca["Representation"].eq(representation)]
    for strategy in MODEL_ORDER + ["Treat all", "Treat none"]:
        data = dca_data.loc[dca_data["Strategy"].eq(strategy)].sort_values("Threshold")
        if data.empty: continue
        linestyle = "--" if strategy == "Treat all" else ":" if strategy == "Treat none" else "-"
        color = MODEL_COLORS.get(strategy, CNS["gray"])
        ax.plot(data["Threshold"], data["Net_Benefit"], color=color, linestyle=linestyle, linewidth=2.0, label=strategy)
    ax.set_xlabel("Predicted-risk threshold"); ax.set_ylabel("Net benefit")
    ax.set_title(f"Supplementary survival DCA: {representation}, 10 years", loc="left", fontweight="bold")
    ax.grid(color=CNS["light_gray"], linewidth=0.8); ax.legend(frameon=False, fontsize=10.5, ncol=2); clean_axes(ax, grid_axis="both"); fig.tight_layout()
    save_figure(fig, f"supplement_result4_{safe}_10y_survival_dca"); plt.show()


In [ ]:
# 5. Output manifest and explicit estimand audit
estimand_audit = pd.DataFrame([
    {"Metric":"Uno IPCW C-index", "Tau_Years":UNO_TAU, "Role":"primary", "Censoring_Distribution_Estimated_From":"locked training set"},
    {"Metric":"IPCW integrated Brier score", "Tau_Years":10.0, "Role":"secondary", "Censoring_Distribution_Estimated_From":"locked training set"},
    {"Metric":"Calibration by test-risk deciles with KM observed risk", "Tau_Years":10.0, "Role":"secondary", "Censoring_Distribution_Estimated_From":"not applicable"},
    {"Metric":"Standard survival decision curve analysis", "Tau_Years":10.0, "Role":"secondary", "Censoring_Distribution_Estimated_From":"dcurves internal survival method"},
])
estimand_audit["Evaluation_Population"] = "same locked test set for all models"
estimand_audit["Compared_Models"] = " | ".join(MODEL_ORDER)
estimand_audit["Bootstrap_Design"] = "paired within representation where differences are reported"
estimand_audit["Bootstrap_N"] = BOOTSTRAP_N
estimand_audit["Prediction_Models_Refit_During_Bootstrap"] = False
estimand_audit["Cohort_ID"] = cohort_id
estimand_audit["Analysis_Role"] = analysis_role
estimand_audit_path = RESULT4_UNO_DIR / "result4_uno_estimand_audit.csv"
estimand_audit.to_csv(estimand_audit_path, index=False)

stems = [
    "figure4A_all_models_uno_ipcw_cns_final",
    "figure4B_difference_in_discrimination_relative_to_cox_uno_cns_final",
    "supplement_result4_all_models_10y_ipcw_ibs",
] + [
    f"supplement_result4_{re.sub(r'[^a-z0-9]+', '_', representation.lower()).strip('_')}_10y_{suffix}"
    for representation in REP_ORDER for suffix in ["calibration", "survival_dca"]
]
paths = [FIGURE_DIR / f"{stem}.{extension}" for stem in stems for extension in ["png", "pdf", "svg"]]
manifest = pd.DataFrame({
    "Path": [str(path) for path in paths],
    "Exists": [path.exists() for path in paths],
    "Size_Bytes": [path.stat().st_size if path.exists() else np.nan for path in paths],
})
manifest_path = RESULT4_UNO_DIR / "result4_cns_final_figure_manifest.csv"
manifest.to_csv(manifest_path, index=False)

print("Result 4 final complete.")
print("Prediction models were trained/loaded before evaluation; none was refitted inside the paired test bootstrap.")
print("Estimand audit:", estimand_audit_path)
print("Manifest:", manifest_path)
display(estimand_audit)
display(manifest)


In [ ]:
cohort_id, analysis_role = _pipeline_context(pipeline_manifest)
context_table = pd.DataFrame([{
    "Result":"Result4", "Cohort_ID":cohort_id, "Analysis_Role":analysis_role,
    "Representation_Count":3, "Model_Count":4,
    "Models":"Cox PH | Elastic-net Cox | XGBoost survival | DeepSurv",
    "Primary_Metric":"10-year Uno IPCW C-index",
}])
atomic_write_csv(context_table, RESULT4_UNO_DIR/"result4_analysis_context.csv", index=False)
atomic_write_json({
    "status":"success", "completed_at":str(pd.Timestamp.now()),
    "pipeline_manifest_sha256":current_pipeline_sha256(),
    "cohort_id":cohort_id, "analysis_role":analysis_role,
    "models":["Cox PH","Elastic-net Cox","XGBoost survival","DeepSurv"],
    "representations":[feature for _,feature in REPRESENTATIONS],
    "expanded_panel_included":False,
    "standard_deepsurv_only":True, "primary_metric":"10-year Uno IPCW C-index",
    "secondary_10y_metrics":["IPCW IBS","calibration","survival DCA"],
    "test_used_for_tuning":False,
}, RESULT4_UNO_DIR/"SUCCESS.json")
